In [ ]:
import json
import time
from pathlib import Path

In [ ]:
# Import kaggle_benchmarks as kbench for using free AI quota provided by Kaggle (10 usd/day - 100 usd/month). 
# This is a temporary solution to avoid the cost of using API as OpenAI.
import kaggle_benchmarks as kbench

# My project uses the "google/gemini-3.6-flash" provided by Kaggle Benchmarks.
llm=kbench.llms["google/gemini-3.6-flash"]

#  You can check the available LLMs by printing the keys of kbench.llms dictionary.
print(list(kbench.llms.keys())) 

In [ ]:
# Call the LLM many times can lead to exceeding the free quota or overloading the LLM,
# so you should consider splitting the files into parts and adding a delay between calls.

PART = 1
TOTAL_PARTS = 1
DELAY_BETWEEN_CALLS = 60.0

MAX_RETRIES = 5

# The topic of the video, which will be used in the prompt to guide the editing process.
VIDEO_TOPIC = ""

def get_total_files(transcript_dir, part=PART, total_parts=TOTAL_PARTS):
    raw_paths = transcript_dir.glob("*_transcript.json")
    all_json_paths = sorted(set([str(p) for p in raw_paths]))
    total_all_files = len(all_json_paths)

    if total_all_files == 0:
        print(f"⚠️ Không tìm thấy file JSON nào trong {transcript_dir}")

    avg_chunk = total_all_files // total_parts
    start_idx = (part - 1) * avg_chunk
    end_idx = total_all_files if part == total_parts else part * avg_chunk

    part_paths = all_json_paths[start_idx:end_idx]

    print(f"🔥 PART {part}/{total_parts} - Transcript dir: {transcript_dir}")
    print(f"📊 Xử lý từ file #{start_idx + 1} đến #{end_idx} (Tổng {len(part_paths)}/{total_all_files} files)")
    print("----------------------------------------------------------------------")
    return part_paths

def build_prompt(event_text: str) -> str:
        return f"""Bạn là chuyên gia tóm lược cho video {VIDEO_TOPIC}. Hãy tóm lược văn bản lời thoại (transcript) sau.

YÊU CẦU:
- Tóm tắt ngắn gọn nhưng đủ các diễn biến chính của toàn bộ video.
- Không thêm giải thích, không đánh số thứ tự, không dùng markdown, không lặp lại nguyên văn cả câu gốc.

Văn bản lời thoại: {event_text}"""

In [ ]:
transcript_dir = Path("")  #folder containing all transcript files
base_output_dir = Path("") #folder to save the output files

@kbench.task(name=f"Summary Transcript Text", description=f"Tóm lược nội dung cho text của toàn transcript")
def summary_transcript_text_task(llm) -> None:
    part_paths = get_total_files(transcript_dir)

    for idx, input_path in enumerate(part_paths, 1):
        filename = Path(input_path).name
        output_path = base_output_dir / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"🎬 [{idx}/{len(part_paths)}] Đang xử lý: {filename}")

        if output_path.resolve() == Path(input_path).resolve():
            raise RuntimeError(f"output_path trùng input_path ({input_path})!")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue

        video_key = next(iter(data), None)
        if video_key is None:
            print("    ⏩ File rỗng. Bỏ qua.")
            continue

        info = data[video_key]
        if info.get("correct_status") != "success":
            print(f"    ⏩ Bỏ qua: correct status group = {info.get('correct_status')}")
            info["transcript_summary_status"] = "skipped"
            info["transcript_summary"] = None
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump({video_key: info}, 
                         f, ensure_ascii=False, indent=2)
            continue
    
        all_events = [e for e in data[video_key].get("events", [])]
        
        total_events = len(all_events)

        if total_events == 0:
            print("    ⏩ File không có events. Ghi kết quả rỗng.")
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump({video_key: {"status": "empty", "events": []}},
                          f, ensure_ascii=False, indent=2)
            continue

        print(f"    📦 Bắt đầu xử lý...")

        transcript = " ".join([(ev.get("text") or "").strip() for ev in all_events])

        if not transcript:
            status = "skipped"
            transcript_summary = ""
        else:
            prompt = build_prompt(transcript)
    
            txt = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    txt = llm.prompt(prompt).strip()
                    break
                except Exception as err:
                    print(f"    ⚠️ Lỗi transcript (lần {attempt}/{MAX_RETRIES}): {err}")
                    if attempt < MAX_RETRIES:
                        time.sleep(attempt * 10.0)
    
            if txt:
                ratio = len(txt) / len(transcript)
                if ratio > 0.3 or ratio < 0.02:
                    print(f"    🚩 Tỉ lệ tóm tắt/gốc = {ratio:.1%}\n    - SUMMARY: {txt}")                    
                print(f"    Transcript summary: SUCCESS. {len(transcript)} -> {len(txt)}")
                status = "success"
                transcript_summary = txt
            else:
                status = "failed"
                transcript_summary = ""
                print(f"    ❌ Transcript summary: FAILED")
            
        output_data = {
            video_key: {
                "transcript_summary_status": status,
                "transcript_summary": transcript_summary,
                "event_summary_status": info.get("summary_status", "unknown"),
                "correct_status": info.get("correct_status", "unknown"),
                "group_status": info.get("group_status", "unknown"),
                "events": all_events
                }
            }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
  
        print(f"    ✅ {filename}: status={status} -> {output_path}")

    print(f"\n⚔️ HOÀN THÀNH: PART {PART}/{TOTAL_PARTS}!")
    kbench.assertions.assert_true(True, expectation=f"Done Part {PART}")

In [ ]:
# Run the task with the specified LLM
summary_transcript_text_task.run(llm=llm)